# EEG Trial And Label Verification

This notebook reproduces the trial split logic in `eeg.py`, visualizes the trial boundaries on the raw signal, and checks whether window labels produced by Braindecode match the expected phase labels. Run it in the `mne` conda environment.

In [ ]:
from pathlib import Path
import os
import warnings

import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
import yaml
from IPython.display import display
from braindecode.datasets import create_from_mne_raw

In [ ]:
def detect_project_root():
    candidates = [
        Path.cwd(),
        Path('/Users/hpyi/SAYGB/26Codes/NewRO25'),
        Path('/content/NewRO25'),
    ]
    env_root = os.environ.get('ROBIO_PROJECT_ROOT')
    if env_root:
        candidates.insert(0, Path(env_root).expanduser())
    for candidate in candidates:
        root = candidate.expanduser().resolve()
        if (root / 'eeg.py').exists() and (root / 'config.yaml').exists():
            return root
    raise FileNotFoundError('Cannot find project root containing eeg.py and config.yaml.')


PROJECT_ROOT = detect_project_root()
DATA_DIR = PROJECT_ROOT / 'PPEEG'
CONFIG = yaml.safe_load((PROJECT_ROOT / 'config.yaml').read_text())
FILES = sorted(path for path in DATA_DIR.glob('*A1*.fif') if path.is_file())

TARGET_INDEX = 0
TARGET_DESC = 'Stimulus/S  5'
REST_DUR = 20
FLEX_DUR = 10
EXT_DUR = 5
WINDOW_SIZE_SAMPLES = int(CONFIG['window_size_samples'])
WINDOW_STRIDE_SAMPLES = int(CONFIG['window_stride_samples'])
TRIAL_ID_TO_PLOT = 0
PICKS = None

if not FILES:
    raise FileNotFoundError(f'No A1 FIF files found in {DATA_DIR}')
if TARGET_INDEX < 0 or TARGET_INDEX >= len(FILES):
    raise IndexError(f'TARGET_INDEX {TARGET_INDEX} is out of range for {len(FILES)} files')

TARGET_FILE = FILES[TARGET_INDEX]
print('Project root:', PROJECT_ROOT)
print('Selected file:', TARGET_FILE)
print('Window size:', WINDOW_SIZE_SAMPLES)
print('Window stride:', WINDOW_STRIDE_SAMPLES)

In [ ]:
EVENT_LABELS = ('Rest', 'Elbow_Flexion', 'Elbow_Extension')
EVENT_MAPPING = {'Rest': 0, 'Elbow_Flexion': 1, 'Elbow_Extension': 2}
PHASE_COLORS = {
    'Rest': '#d9ead3',
    'Elbow_Flexion': '#fce5cd',
    'Elbow_Extension': '#cfe2f3',
}


def collect_original_annotations(raw, start_s, stop_s):
    rows = []
    for onset, duration, desc in zip(raw.annotations.onset, raw.annotations.duration, raw.annotations.description):
        ann_start = float(onset)
        ann_stop = float(onset + duration)
        overlaps = ann_stop > start_s and ann_start < stop_s
        if overlaps:
            rows.append(
                {
                    'description': desc,
                    'start': ann_start,
                    'stop': ann_stop,
                    'duration': float(duration),
                }
            )
    return rows


def build_trial_records(raw, matched_onsets, rest_duration, flex_duration, ext_duration):
    sfreq = float(raw.info['sfreq'])
    raw_end_time = raw.times[-1] + (1.0 / sfreq)
    trial_records = []
    prev_ext_end = None

    for trial_id, flex_time in enumerate(matched_onsets):
        rest_onset = (
            flex_time - rest_duration
            if prev_ext_end is None
            else min(prev_ext_end, flex_time - rest_duration)
        )
        flex_onset = flex_time
        ext_onset = flex_time + flex_duration
        trial_end = ext_onset + ext_duration
        prev_ext_end = trial_end

        valid = rest_onset >= 0 and trial_end <= raw_end_time
        record = {
            'trial_id': int(trial_id),
            'rest_onset': float(rest_onset),
            'flex_onset': float(flex_onset),
            'ext_onset': float(ext_onset),
            'trial_end': float(trial_end),
            'valid': bool(valid),
        }

        if valid:
            crop_end = max(rest_onset, trial_end - (1.0 / sfreq))
            trial_raw = raw.copy().crop(tmin=rest_onset, tmax=crop_end)
            trial_raw.set_annotations(
                mne.Annotations(
                    onset=[0.0, flex_onset - rest_onset, ext_onset - rest_onset],
                    duration=[rest_duration, flex_duration, ext_duration],
                    description=list(EVENT_LABELS),
                )
            )
            record['raw'] = trial_raw
            record['original_annotations'] = collect_original_annotations(raw, rest_onset, trial_end)
            local_flex_onset = float(flex_onset - rest_onset)
            local_ext_onset = float(ext_onset - rest_onset)
            record['local_bounds'] = [
                {'label': 'Rest', 'start': 0.0, 'stop': local_flex_onset, 'duration': local_flex_onset},
                {
                    'label': 'Elbow_Flexion',
                    'start': local_flex_onset,
                    'stop': local_ext_onset,
                    'duration': float(flex_duration),
                },
                {
                    'label': 'Elbow_Extension',
                    'start': local_ext_onset,
                    'stop': float(trial_end - rest_onset),
                    'duration': float(ext_duration),
                },
            ]
            trial_records.append(record)

    return trial_records


def create_windows_dataset_from_trials(trial_records, subject_id, window_size_samples, window_stride_samples):
    parts = [record['raw'] for record in trial_records]
    descriptions = [
        {
            'event_code': [0, 1, 2],
            'subject': subject_id,
            'trial_id': int(record['trial_id']),
        }
        for record in trial_records
    ]
    return create_from_mne_raw(
        parts,
        trial_start_offset_samples=0,
        trial_stop_offset_samples=0,
        window_size_samples=window_size_samples,
        window_stride_samples=window_stride_samples,
        drop_last_window=False,
        descriptions=descriptions,
        mapping=EVENT_MAPPING,
    )


def build_trial_summary(trial_records):
    rows = []
    for record in trial_records:
        rows.append(
            {
                'trial_id': record['trial_id'],
                'rest_onset': record['rest_onset'],
                'flex_onset': record['flex_onset'],
                'ext_onset': record['ext_onset'],
                'trial_end': record['trial_end'],
                'trial_duration': record['trial_end'] - record['rest_onset'],
                'local_flex_onset': record['local_bounds'][1]['start'],
                'local_ext_onset': record['local_bounds'][2]['start'],
                'local_trial_end': record['local_bounds'][2]['stop'],
            }
        )
    return pd.DataFrame(rows)


def expected_target_for_window(local_start_s, local_stop_s, local_bounds):
    center_s = (local_start_s + local_stop_s) / 2.0
    for phase in local_bounds:
        if phase['start'] <= center_s < phase['stop']:
            return EVENT_MAPPING[phase['label']]
    if np.isclose(center_s, local_bounds[-1]['stop']):
        return EVENT_MAPPING[local_bounds[-1]['label']]
    raise ValueError(f'Window center {center_s:.6f}s is outside annotated trial bounds.')


def build_window_table(windows_dataset, trial_records):
    trial_lookup = {record['trial_id']: record for record in trial_records}
    rows = []

    for sub_dataset in windows_dataset.datasets:
        meta = sub_dataset.windows.metadata.copy().reset_index(drop=True)
        trial_id = int(sub_dataset.description['trial_id'])
        record = trial_lookup[trial_id]
        sfreq = float(record['raw'].info['sfreq'])
        first_samp = int(record['raw'].first_samp)

        for _, row in meta.iterrows():
            local_start_s = (int(row['i_start_in_trial']) - first_samp) / sfreq
            local_stop_s = (int(row['i_stop_in_trial']) - first_samp) / sfreq
            expected_target = expected_target_for_window(local_start_s, local_stop_s, record['local_bounds'])
            rows.append(
                {
                    'trial_id': trial_id,
                    'window_index': int(row['i_window_in_trial']),
                    'start_s': local_start_s,
                    'stop_s': local_stop_s,
                    'center_s': (local_start_s + local_stop_s) / 2.0,
                    'target': int(row['target']),
                    'target_name': EVENT_LABELS[int(row['target'])],
                    'expected_target': int(expected_target),
                    'expected_name': EVENT_LABELS[int(expected_target)],
                    'label_ok': int(row['target']) == int(expected_target),
                }
            )

    return pd.DataFrame(rows)


def plot_trial_on_raw(raw, record, picks=None, padding=2.0, figsize=(14, 6)):
    start_time = max(0.0, record['rest_onset'] - padding)
    stop_time = min(raw.times[-1], record['trial_end'] + padding)
    start_sample = raw.time_as_index(start_time)[0]
    stop_sample = raw.time_as_index(stop_time)[0]

    if picks is None:
        picks = raw.ch_names[: min(5, len(raw.ch_names))]

    data = raw.get_data(picks=picks, start=start_sample, stop=stop_sample)
    times = raw.times[start_sample:stop_sample]
    offset_step = max(float(data.std(axis=1).max()) * 6.0, 1e-6) if data.size else 1.0

    fig, ax = plt.subplots(figsize=figsize)
    for idx, ch_name in enumerate(picks):
        ax.plot(times, data[idx] + idx * offset_step, linewidth=1.0, label=ch_name)

    phase_bounds = [
        ('Rest', record['rest_onset'], record['flex_onset']),
        ('Elbow_Flexion', record['flex_onset'], record['ext_onset']),
        ('Elbow_Extension', record['ext_onset'], record['trial_end']),
    ]
    for label, start_s, stop_s in phase_bounds:
        ax.axvspan(start_s, stop_s, color=PHASE_COLORS[label], alpha=0.35, label=label)

    ax.axvline(record['flex_onset'], color='black', linestyle='--', linewidth=1.0, label=TARGET_DESC)
    for ann in record['original_annotations']:
        ax.axvline(ann['start'], color='#666666', linestyle=':', linewidth=0.9)
    handles, labels = ax.get_legend_handles_labels()
    dedup = dict(zip(labels, handles))
    ax.legend(dedup.values(), dedup.keys(), loc='upper right')
    ax.set_title(f'Trial {record["trial_id"]} on raw: {TARGET_FILE.name}')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Amplitude + offset')
    ax.grid(True, alpha=0.2)
    plt.show()


def plot_trial_cropped(record, picks=None, figsize=(14, 5)):
    trial_raw = record['raw']
    if picks is None:
        picks = trial_raw.ch_names[: min(5, len(trial_raw.ch_names))]

    data = trial_raw.get_data(picks=picks)
    times = trial_raw.times
    offset_step = max(float(data.std(axis=1).max()) * 6.0, 1e-6) if data.size else 1.0

    fig, ax = plt.subplots(figsize=figsize)
    for idx, ch_name in enumerate(picks):
        ax.plot(times, data[idx] + idx * offset_step, linewidth=1.0, label=ch_name)

    for phase in record['local_bounds']:
        ax.axvspan(phase['start'], phase['stop'], color=PHASE_COLORS[phase['label']], alpha=0.35, label=phase['label'])

    handles, labels = ax.get_legend_handles_labels()
    dedup = dict(zip(labels, handles))
    ax.legend(dedup.values(), dedup.keys(), loc='upper right')
    ax.set_title(f'Cropped trial {record["trial_id"]} with relabeled phases')
    ax.set_xlabel('Time within trial (s)')
    ax.set_ylabel('Amplitude + offset')
    ax.grid(True, alpha=0.2)
    plt.show()


def plot_window_labels(window_table, record, trial_id, figsize=(14, 4)):
    trial_windows = window_table.loc[window_table['trial_id'] == trial_id].copy()
    if trial_windows.empty:
        raise ValueError(f'No windows found for trial {trial_id}')

    fig, ax = plt.subplots(figsize=figsize)
    for phase in record['local_bounds']:
        ax.axvspan(phase['start'], phase['stop'], color=PHASE_COLORS[phase['label']], alpha=0.25)

    for _, row in trial_windows.iterrows():
        color = PHASE_COLORS[row['target_name']]
        alpha = 0.9 if row['label_ok'] else 0.35
        ax.plot([row['start_s'], row['stop_s']], [row['window_index'], row['window_index']], color=color, linewidth=4, alpha=alpha)

    ax.set_title(f'Window labels for trial {trial_id}')
    ax.set_xlabel('Time within trial (s)')
    ax.set_ylabel('Window index')
    ax.grid(True, alpha=0.2)
    plt.show()


def build_original_annotation_table(trial_records):
    rows = []
    for record in trial_records:
        for ann in record['original_annotations']:
            rows.append(
                {
                    'trial_id': record['trial_id'],
                    'description': ann['description'],
                    'start_s': ann['start'],
                    'stop_s': ann['stop'],
                    'duration_s': ann['duration'],
                }
            )
    return pd.DataFrame(rows)

In [ ]:
warnings.filterwarnings('ignore', message='Limited .* annotation\(s\) that were expanding outside the data range.')
warnings.filterwarnings('ignore', message='Drop bad windows only has an effect if mne epochs are created.*')
warnings.filterwarnings('ignore', message='Using reject or picks or flat or dropping bad windows means mne Epochs are created.*')
mne.set_log_level('WARNING')
raw = mne.io.read_raw_fif(TARGET_FILE, preload=True)
subject_id = TARGET_FILE.name[:4]
matched_onsets = [
    onset
    for onset, desc in zip(raw.annotations.onset, raw.annotations.description)
    if desc.strip() == TARGET_DESC.strip()
]
trial_records = build_trial_records(
    raw,
    matched_onsets,
    rest_duration=REST_DUR,
    flex_duration=FLEX_DUR,
    ext_duration=EXT_DUR,
)
trial_summary = build_trial_summary(trial_records)

print('Matched onsets:', len(matched_onsets))
print('Retained trials:', len(trial_records))
print('Subject:', subject_id)
display(trial_summary.head(len(trial_summary)))

In [ ]:
original_annotation_table = build_original_annotation_table(trial_records)
record = next(record for record in trial_records if record['trial_id'] == TRIAL_ID_TO_PLOT)
plot_trial_on_raw(raw, record, picks=PICKS, padding=2.0)
plot_trial_cropped(record, picks=PICKS)
display(pd.DataFrame(record['local_bounds']))
display(pd.DataFrame(record['original_annotations']))

In [ ]:
for record in trial_records:
    print(f'===== Trial {record["trial_id"]} =====')
    plot_trial_on_raw(raw, record, picks=PICKS, padding=1.0)
    plot_trial_cropped(record, picks=PICKS)
    display(pd.DataFrame(record['local_bounds']))
    display(pd.DataFrame(record['original_annotations']))

In [ ]:
windows_dataset = create_windows_dataset_from_trials(
    trial_records,
    subject_id=subject_id,
    window_size_samples=WINDOW_SIZE_SAMPLES,
    window_stride_samples=WINDOW_STRIDE_SAMPLES,
)
window_table = build_window_table(windows_dataset, trial_records)

print('Total windows:', len(window_table))
print('Mismatched windows:', int((~window_table['label_ok']).sum()))
display(window_table.head(20))
display(window_table.groupby(['trial_id', 'target_name']).size().unstack(fill_value=0))

In [ ]:
plot_window_labels(window_table, record, trial_id=TRIAL_ID_TO_PLOT)
display(window_table.loc[window_table['trial_id'] == TRIAL_ID_TO_PLOT].head(30))

In [ ]:
assert window_table['label_ok'].all(), 'Found window labels that do not match the expected phase label.'
print('All window labels match the expected trial phase labels.')